# GNN (low-level): extended 8-node graph


**Graph:** **8 nodes**, 4 features each, **fully connected** (directed). Still **no separate `data.u` vector** — everything is node input.

| Node | Meaning | Features |
|------|---------|----------|
| 0–4 | Higgs, lead jet, sublead jet, two photons | pT, η, φ, M (γ: M=0) |
| 5 | Dijet system | pT_jj, η_jj, φ_jj, M_jj |
| 6 | γγ summary | ΣpT(γ), η_yybb, Δφ_yybb, 0 |
| 7 | Angular / CP scalars | cosθ*, cosθ1, cosθ2, Δφ_bb |

**Models (more expressive than the original 3-layer GCN/GAT):**
1. **Deep residual GCN** — linear input projection → 4× `GCNConv` with **residual** adds + `LayerNorm`, wider hidden dim (96), deeper MLP head.
2. **Graph Transformer** — 3× `TransformerConv` (multi-head self-attention on the graph) + `GELU` + deeper classifier.

Compare with `04_graph_neural_network_classification.ipynb` (5 nodes + 10 globals) to see effect of topology and capacity.

**Requirements:** `torch`, `torch-geometric` (see `requirements.txt`).

**Data:** All node inputs come from `ForAnalysis/1d` (not `2d`, which is unfilled in March 2026 files). The extra nodes are **composite** rows built from dijet / diphoton / angular columns in the same table.

In [ ]:
import pandas as pd
import numpy as np
import h5py
import matplotlib.pyplot as plt
from plot_style import apply_publication_style
apply_publication_style()
import time
import os
from sklearn.metrics import roc_curve, auc, accuracy_score, precision_score, recall_score, f1_score

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCNConv, TransformerConv, global_mean_pool, global_max_pool

np.random.seed(42)
torch.manual_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

os.makedirs('plots', exist_ok=True)
os.makedirs('models', exist_ok=True)
os.makedirs('metrics', exist_ok=True)

## 1. Load Data

In [ ]:
def load_dataset(filepath):
    with h5py.File(filepath, 'r') as f:
        df = pd.DataFrame.from_records(f['ForAnalysis/1d'][:])
    return df

sm_df = load_dataset('datasets/new_Input_bbyy_SMEFT_SM_4thMarch_2026.h5')
cbbim_df = load_dataset('datasets/new_Input_bbyy_SMEFT_cbbim_4thMarch_2026.h5')
cbgim_df = load_dataset('datasets/new_Input_bbyy_SMEFT_cbgim_4thMarch_2026.h5')
cbhim_df = load_dataset('datasets/new_Input_bbyy_SMEFT_cbhim_4thMarch_2026.h5')
chbtil_df = load_dataset('datasets/new_Input_bbyy_SMEFT_chbtil_4thMarch_2026.h5')
chgtil_df = load_dataset('datasets/new_Input_bbyy_SMEFT_chgtil_4thMarch_2026.h5')
ctbim_df = load_dataset('datasets/new_Input_bbyy_SMEFT_ctbim_4thMarch_2026.h5')

print(f"SM events: {len(sm_df):,}")
for name, df in [('cbbim', cbbim_df), ('cbgim', cbgim_df), ('cbhim', cbhim_df),
                 ('chbtil', chbtil_df), ('chgtil', chgtil_df), ('ctbim', ctbim_df)]:
    print(f"  {name}: {len(df):,}")

bsm_datasets = {
    'cbbim': cbbim_df, 'cbgim': cbgim_df, 'cbhim': cbhim_df,
    'chbtil': chbtil_df, 'chgtil': chgtil_df, 'ctbim': ctbim_df,
}

## 2. Events → graphs (8 nodes, no `data.u`)

- **Edges:** fully connected directed (8×7 = 56 edges per graph)
- **Standardization:** per-feature mean/std from the **training** split only (applied to all 8×4 entries)

In [ ]:
N_NODES = 8
NODE_NAMES = [
    'Higgs', 'LeadJet', 'SubLeadJet', 'LeadPhoton', 'SubLeadPhoton',
    'DijetSystem', 'PhotonPair', 'AngularCP',
]

def fully_connected_edge_index(n):
    return torch.tensor(
        [[i, j] for i in range(n) for j in range(n) if i != j],
        dtype=torch.long).t().contiguous()

def event_to_graph(row, label):
    """8 nodes × 4 features: 5 objects + jj + γγ summary + angular block."""
    lp_pt = float(row['LeadPhoton_pT'])
    slp_pt = float(row['SubLeadPhoton_pT'])
    x = torch.tensor([
        [row['Higgs_pT'], row['Higgs_Eta'], row['Higgs_Phi'], row['Higgs_Mass']],
        [row['LeadJet_pT'], row['LeadJet_Eta'], row['LeadJet_Phi'], row['LeadJet_M']],
        [row['SubLeadJet_pT'], row['SubLeadJet_Eta'], row['SubLeadJet_Phi'], row['SubLeadJet_M']],
        [row['LeadPhoton_pT'], row['LeadPhoton_Eta'], row['LeadPhoton_Phi'], 0.0],
        [row['SubLeadPhoton_pT'], row['SubLeadPhoton_Eta'], row['SubLeadPhoton_Phi'], 0.0],
        [row['pT_jj'], row['Eta_jj'], row['Phi_jj'], row['M_jj']],
        [lp_pt + slp_pt, row['Eta_yybb'], row['DPhi_yybb'], 0.0],
        [row['cosThetaStar'], row['costheta1'], row['costheta2'], row['DPhi_bb']],
    ], dtype=torch.float32)
    return Data(
        x=x,
        edge_index=fully_connected_edge_index(N_NODES),
        y=torch.tensor([float(label)], dtype=torch.float),
    )


def _get_event_type(row):
    if row.get('is_ZHEvent', False):
        return 'ZH'
    if row.get('is_HHEvent', False):
        return 'HH'
    if row.get('is_SingleHiggsEvent', False):
        return 'Single H'
    return 'Other'


def create_graph_dataset(sm_df, bsm_df, balance=True):
    graphs = []
    if balance:
        sm_copy = sm_df.copy()
        bsm_copy = bsm_df.copy()
        sm_copy['_event_type'] = sm_copy.apply(_get_event_type, axis=1)
        bsm_copy['_event_type'] = bsm_copy.apply(_get_event_type, axis=1)
        sm_samples, bsm_samples = [], []
        for etype in ['ZH', 'HH', 'Single H', 'Other']:
            sm_sub = sm_copy[sm_copy['_event_type'] == etype]
            bsm_sub = bsm_copy[bsm_copy['_event_type'] == etype]
            n_min = min(len(sm_sub), len(bsm_sub))
            if n_min > 0:
                sm_samples.append(sm_sub.sample(n=n_min, random_state=42))
                bsm_samples.append(bsm_sub.sample(n=n_min, random_state=42))
        if sm_samples and bsm_samples:
            sm_sample = pd.concat(sm_samples, ignore_index=True)
            bsm_sample = pd.concat(bsm_samples, ignore_index=True)
        else:
            n_min = min(len(sm_df), len(bsm_df))
            sm_sample = sm_df.sample(n=n_min, random_state=42)
            bsm_sample = bsm_df.sample(n=n_min, random_state=42)
    else:
        sm_sample = sm_df
        bsm_sample = bsm_df
    for _, row in sm_sample.iterrows():
        graphs.append(event_to_graph(row, 0))
    for _, row in bsm_sample.iterrows():
        graphs.append(event_to_graph(row, 1))
    return graphs


def mean_std_from_graphs(graph_list):
    xs = torch.cat([g.x for g in graph_list], dim=0)
    m = xs.mean(dim=0)
    s = torch.clamp(xs.std(dim=0), min=1e-4)
    return m, s


def apply_std(graph_list, m, s):
    for g in graph_list:
        g.x = (g.x - m) / s


def prepare_data(sm_df, bsm_df, batch_size=64):
    graph_dataset = create_graph_dataset(sm_df, bsm_df)
    np.random.shuffle(graph_dataset)
    n_total = len(graph_dataset)
    n_train = int(0.7 * n_total)
    n_val = int(0.1 * n_total)
    train_ds = graph_dataset[:n_train]
    val_ds = graph_dataset[n_train:n_train + n_val]
    test_ds = graph_dataset[n_train + n_val:]
    xm, xs = mean_std_from_graphs(train_ds)
    apply_std(train_ds, xm, xs)
    apply_std(val_ds, xm, xs)
    apply_std(test_ds, xm, xs)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False)
    return train_loader, val_loader, test_loader, n_total

print('Low-level graph helpers ready.')

In [ ]:
g0 = event_to_graph(sm_df.iloc[0], 0)
print('Sample graph (before standardization):')
print(f'  Nodes: {g0.x.shape[0]} ({", ".join(NODE_NAMES)})')
print(f'  Node dim: {g0.x.shape[1]}, edges: {g0.edge_index.shape[1]}')
print(f'  Has global u: {getattr(g0, "u", None) is not None}')
for i, name in enumerate(NODE_NAMES):
    print(f'  [{i}] {name}:', g0.x[i].tolist())

## 3. Models — deep residual GCN & graph transformer

- **DeepResidualGCN:** `Linear(4→96)` then 4× `GCNConv(96→96)` with **residual** `x ← x + conv(x)` after each block, `LayerNorm`, classifier `192 → 128 → 64 → 1`.
- **GraphTransformer:** 3× `TransformerConv` (4 heads, then 4 heads, then 1 head) with `GELU`, `LayerNorm`, classifier `96 → 128 → 64 → 1`.

Training uses `BCEWithLogitsLoss` + grad clipping.

In [ ]:
class DeepResidualGCNClassifier(nn.Module):
    """Wider + deeper GCN with residual connections between layers (same hidden dim)."""

    def __init__(self, node_features=4, hidden_dim=96, num_layers=4, dropout=0.25):
        super().__init__()
        self.input_proj = nn.Linear(node_features, hidden_dim)
        self.convs = nn.ModuleList()
        self.norms = nn.ModuleList()
        for _ in range(num_layers):
            self.convs.append(GCNConv(hidden_dim, hidden_dim))
            self.norms.append(nn.LayerNorm(hidden_dim))
        self.num_layers = num_layers
        self.dropout = dropout
        cdim = hidden_dim * 2
        self.classifier = nn.Sequential(
            nn.Linear(cdim, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1),
        )

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        x = self.input_proj(x)
        for i in range(self.num_layers):
            h = self.convs[i](x, edge_index)
            h = self.norms[i](h)
            h = F.relu(h)
            h = F.dropout(h, p=self.dropout, training=self.training)
            x = x + h
        xm = global_mean_pool(x, batch)
        xx = global_max_pool(x, batch)
        return self.classifier(torch.cat([xm, xx], dim=1)).squeeze(-1)


class GraphTransformerClassifier(nn.Module):
    """Multi-head self-attention on graph nodes (TransformerConv stack)."""

    def __init__(self, node_features=4, hidden_dim=32, heads=4, num_layers=3, dropout=0.25):
        super().__init__()
        self.convs = nn.ModuleList()
        self.norms = nn.ModuleList()
        d_in = node_features
        d_mid = hidden_dim * heads
        self.convs.append(
            TransformerConv(d_in, hidden_dim, heads=heads, dropout=dropout, concat=True)
        )
        self.norms.append(nn.LayerNorm(d_mid))
        for _ in range(num_layers - 2):
            self.convs.append(
                TransformerConv(d_mid, hidden_dim, heads=heads, dropout=dropout, concat=True)
            )
            self.norms.append(nn.LayerNorm(d_mid))
        out_dim = 64
        self.convs.append(
            TransformerConv(d_mid, out_dim, heads=1, concat=False, dropout=dropout)
        )
        self.norms.append(nn.LayerNorm(out_dim))
        self.num_layers = num_layers
        self.dropout = dropout
        self.final_dim = out_dim
        cdim = out_dim * 2
        self.classifier = nn.Sequential(
            nn.Linear(cdim, 128),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(128, 64),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1),
        )

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        for i in range(self.num_layers):
            x = self.convs[i](x, edge_index)
            x = self.norms[i](x)
            x = F.gelu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
        xm = global_mean_pool(x, batch)
        xx = global_max_pool(x, batch)
        return self.classifier(torch.cat([xm, xx], dim=1)).squeeze(-1)


m1 = DeepResidualGCNClassifier().to(device)
m2 = GraphTransformerClassifier().to(device)
print(f'DeepResidualGCN params: {sum(p.numel() for p in m1.parameters()):,}')
print(f'GraphTransformer   params: {sum(p.numel() for p in m2.parameters()):,}')
del m1, m2

## 4. Training & evaluation

In [ ]:
def train_gnn(model, train_loader, val_loader, epochs=100, lr=0.001, patience=15):
    criterion = nn.BCEWithLogitsLoss()
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, patience=5, factor=0.5)
    metrics = {k: [] for k in ['epoch', 'train_loss', 'train_acc', 'val_loss', 'val_acc', 'epoch_time', 'total_time']}
    best_val, best_state, bad = float('inf'), None, 0
    t0 = time.time()
    for epoch in range(epochs):
        t_ep = time.time()
        model.train()
        tr_loss = tr_ok = tr_n = 0
        for batch in train_loader:
            batch = batch.to(device)
            opt.zero_grad()
            out = model(batch)
            loss = criterion(out, batch.y.float())
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            tr_loss += loss.item() * batch.num_graphs
            pr = torch.sigmoid(out)
            tr_ok += ((pr > 0.5).float() == batch.y).sum().item()
            tr_n += batch.num_graphs
        tr_loss /= tr_n
        model.eval()
        va_loss = va_ok = va_n = 0
        with torch.no_grad():
            for batch in val_loader:
                batch = batch.to(device)
                out = model(batch)
                va_loss += criterion(out, batch.y.float()).item() * batch.num_graphs
                pr = torch.sigmoid(out)
                va_ok += ((pr > 0.5).float() == batch.y).sum().item()
                va_n += batch.num_graphs
        va_loss /= va_n
        sched.step(va_loss)
        if va_loss < best_val:
            best_val, best_state, bad = va_loss, {k: v.cpu().clone() for k, v in model.state_dict().items()}, 0
        else:
            bad += 1
        ep_time = time.time() - t_ep
        tot_time = time.time() - t0
        metrics['epoch'].append(epoch + 1)
        metrics['train_loss'].append(tr_loss)
        metrics['train_acc'].append(tr_ok / tr_n)
        metrics['val_loss'].append(va_loss)
        metrics['val_acc'].append(va_ok / va_n)
        metrics['epoch_time'].append(ep_time)
        metrics['total_time'].append(tot_time)
        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(f"Epoch {epoch+1:3d}/{epochs} | train {tr_loss:.4f} acc {tr_ok/tr_n:.4f} | val {va_loss:.4f} acc {va_ok/va_n:.4f}")
        if bad >= patience:
            print(f'Early stopping at epoch {epoch+1}')
            break
    if best_state is not None:
        model.load_state_dict(best_state)
    return metrics, time.time() - t0


def bootstrap_auc(y_true, probs, n_bootstrap=1000, random_state=42):
    rng = np.random.default_rng(random_state)
    n = len(y_true)
    aucs = []
    for _ in range(n_bootstrap):
        idx = rng.choice(n, size=n, replace=True)
        yb, pb = y_true[idx], probs[idx]
        if len(np.unique(yb)) < 2:
            continue
        fpr, tpr, _ = roc_curve(yb, pb)
        aucs.append(auc(fpr, tpr))
    aucs = np.array(aucs)
    return aucs.mean(), aucs.std(), np.percentile(aucs, 2.5), np.percentile(aucs, 97.5)


def evaluate_gnn(model, test_loader, model_name):
    model.eval()
    probs, labels = [], []
    t_inf = time.time()
    with torch.no_grad():
        for batch in test_loader:
            batch = batch.to(device)
            p = torch.sigmoid(model(batch)).cpu().numpy()
            probs.extend(np.atleast_1d(p).tolist())
            labels.extend(batch.y.cpu().numpy().tolist())
    inf_time = time.time() - t_inf
    y = np.array(labels)
    p = np.array(probs)
    pred = (p > 0.5).astype(int)
    fpr, tpr, _ = roc_curve(y, p)
    auc_s = auc(fpr, tpr)
    am, asd, lo, hi = bootstrap_auc(y, p)
    print(f'\n{model_name} Test: Acc={accuracy_score(y, pred):.4f} AUC={am:.4f} ± {asd:.4f} 95% CI [{lo:.4f},{hi:.4f}]')
    return {
        'accuracy': accuracy_score(y, pred), 'precision': precision_score(y, pred, zero_division=0),
        'recall': recall_score(y, pred, zero_division=0), 'f1': f1_score(y, pred, zero_division=0),
        'auc': auc_s, 'auc_mean': am, 'auc_std': asd, 'auc_ci_lower': lo, 'auc_ci_upper': hi,
        'inference_time': inf_time, 'probs': p, 'labels': y, 'fpr': fpr, 'tpr': tpr,
    }

In [ ]:
# Optional: limit operators or epochs for a quick run
EPOCHS = 100
PATIENCE = 15
BATCH_SIZE = 64

all_results = {}

for bsm_name, bsm_df in bsm_datasets.items():
    print(f"\n{'='*70}\n  SM vs {bsm_name} (8-node extended graph)\n{'='*70}")
    t1 = time.time()
    train_loader, val_loader, test_loader, n_g = prepare_data(sm_df, bsm_df, batch_size=BATCH_SIZE)
    print(f'Graphs: {n_g:,} | {N_NODES} nodes, 4 feats (build+split+std in {time.time()-t1:.2f}s)')

    gcn_model = DeepResidualGCNClassifier(node_features=4, hidden_dim=96, num_layers=4).to(device)
    print(f'\nTraining DeepResidualGCN...')
    gcn_metrics, gcn_ttime = train_gnn(gcn_model, train_loader, val_loader, epochs=EPOCHS, patience=PATIENCE)
    gcn_eval = evaluate_gnn(gcn_model, test_loader, f'DeepResidualGCN vs {bsm_name}')

    gat_model = GraphTransformerClassifier(node_features=4, hidden_dim=32, heads=4, num_layers=3).to(device)
    print(f'\nTraining GraphTransformer...')
    gat_metrics, gat_ttime = train_gnn(gat_model, train_loader, val_loader, epochs=EPOCHS, patience=PATIENCE)
    gat_eval = evaluate_gnn(gat_model, test_loader, f'GraphTransformer vs {bsm_name}')

    all_results[bsm_name] = {
        'gcn_model': gcn_model, 'gat_model': gat_model,
        'gcn_metrics': gcn_metrics, 'gat_metrics': gat_metrics,
        'gcn_eval': gcn_eval, 'gat_eval': gat_eval,
        'gcn_train_time': gcn_ttime, 'gat_train_time': gat_ttime,
        'test_loader': test_loader,
    }

    torch.save({
        'model_state_dict': gcn_model.state_dict(), 'model_type': 'DeepResidualGCN',
        'representation': '1d_8x4_no_global', 'n_nodes': N_NODES, 'bsm_operator': bsm_name,
        'auc': gcn_eval['auc'], 'accuracy': gcn_eval['accuracy'],
    }, f'models/gcn_deep_ll_{bsm_name}_classifier.pt')
    torch.save({
        'model_state_dict': gat_model.state_dict(), 'model_type': 'GraphTransformer',
        'representation': '1d_8x4_no_global', 'n_nodes': N_NODES, 'bsm_operator': bsm_name,
        'auc': gat_eval['auc'], 'accuracy': gat_eval['accuracy'],
    }, f'models/gt_ll_{bsm_name}_classifier.pt')

    pd.DataFrame(gcn_metrics).to_csv(f'metrics/gcn_deep_ll_{bsm_name}_training_metrics.csv', index=False)
    pd.DataFrame(gat_metrics).to_csv(f'metrics/gt_ll_{bsm_name}_training_metrics.csv', index=False)

print(f"\n{'='*70}\nTraining complete (DeepResidualGCN + GraphTransformer, 8 nodes).\n{'='*70}")

In [ ]:
for bsm_name, res in all_results.items():
    print(f"SM vs {bsm_name:8s} | DeepGCN AUC: {res['gcn_eval']['auc']:.4f} | GraphTrans AUC: {res['gat_eval']['auc']:.4f}")

summary = pd.DataFrame([{
    'BSM_Operator': k,
    'DeepResidualGCN_AUC': v['gcn_eval']['auc'],
    'GraphTransformer_AUC': v['gat_eval']['auc'],
} for k, v in all_results.items()])
summary.to_csv('metrics/gnn_low_level_comparison.csv', index=False)
summary

## 5. Training curves

In [ ]:
n_bsm = len(all_results)
fig, axes = plt.subplots(n_bsm, 4, figsize=(22, 4 * n_bsm))
if n_bsm == 1:
    axes = axes[np.newaxis, :]

for row, (bsm_name, res) in enumerate(all_results.items()):
    gc, gt = res['gcn_metrics'], res['gat_metrics']
    axes[row, 0].plot(gc['epoch'], gc['train_loss'], label='Train')
    axes[row, 0].plot(gc['epoch'], gc['val_loss'], label='Val')
    axes[row, 0].set_title(f'{bsm_name} DeepGCN loss'); axes[row, 0].legend(); axes[row, 0].grid(True, alpha=0.3)
    axes[row, 1].plot(gc['epoch'], gc['train_acc'], label='Train')
    axes[row, 1].plot(gc['epoch'], gc['val_acc'], label='Val')
    axes[row, 1].set_title(f'{bsm_name} DeepGCN acc'); axes[row, 1].legend(); axes[row, 1].grid(True, alpha=0.3)
    axes[row, 2].plot(gt['epoch'], gt['train_loss'], label='Train')
    axes[row, 2].plot(gt['epoch'], gt['val_loss'], label='Val')
    axes[row, 2].set_title(f'{bsm_name} GraphTrans loss'); axes[row, 2].legend(); axes[row, 2].grid(True, alpha=0.3)
    axes[row, 3].plot(gt['epoch'], gt['train_acc'], label='Train')
    axes[row, 3].plot(gt['epoch'], gt['val_acc'], label='Val')
    axes[row, 3].set_title(f'{bsm_name} GraphTrans acc'); axes[row, 3].legend(); axes[row, 3].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('plots/gnn_ll_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Learned score distributions & ROC

(No global-feature permutation section — this model has no `data.u`.)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
axes = axes.flatten()
for idx, (bsm_name, res) in enumerate(all_results.items()):
    ax = axes[idx]
    p, lab = res['gcn_eval']['probs'], res['gcn_eval']['labels']
    ax.hist(p[lab == 0], bins=30, label='SM', density=True, histtype='step', linewidth=2)
    ax.hist(p[lab == 1], bins=30, label='BSM', density=True, histtype='step', linewidth=2)
    ax.set_title(f'{bsm_name} DeepGCN P(BSM)'); ax.legend()
plt.suptitle('DeepResidualGCN: score distributions (test)')
plt.tight_layout()
plt.savefig('plots/gnn_ll_learned_distribution_gcn.png', dpi=150, bbox_inches='tight')
plt.show()

fig, axes = plt.subplots(2, 3, figsize=(12, 8))
axes = axes.flatten()
for idx, (bsm_name, res) in enumerate(all_results.items()):
    ax = axes[idx]
    p, lab = res['gat_eval']['probs'], res['gat_eval']['labels']
    ax.hist(p[lab == 0], bins=30, label='SM', density=True, histtype='step', linewidth=2)
    ax.hist(p[lab == 1], bins=30, label='BSM', density=True, histtype='step', linewidth=2)
    ax.set_title(f'{bsm_name} GraphTrans P(BSM)'); ax.legend()
plt.suptitle('GraphTransformer: score distributions (test)')
plt.tight_layout()
plt.savefig('plots/gnn_ll_learned_distribution_gat.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(n_bsm, 2, figsize=(14, 5 * n_bsm))
if n_bsm == 1:
    axes = axes[np.newaxis, :]

for row, (bsm_name, res) in enumerate(all_results.items()):
    ge, ga = res['gcn_eval'], res['gat_eval']
    axes[row, 0].plot(ge['fpr'], ge['tpr'], lw=2, label=f"DeepGCN AUC={ge['auc']:.4f}")
    axes[row, 0].plot(ga['fpr'], ga['tpr'], lw=2, label=f"GraphTrans AUC={ga['auc']:.4f}")
    axes[row, 0].plot([0, 1], [0, 1], 'k--', lw=1)
    axes[row, 0].set_xlabel('FPR'); axes[row, 0].set_ylabel('TPR')
    axes[row, 0].set_title(f'ROC SM vs {bsm_name} (low-level)')
    axes[row, 0].legend(); axes[row, 0].grid(True, alpha=0.3)
    axes[row, 1].hist(ge['probs'][ge['labels'] == 0], bins=50, density=True, histtype='step', linewidth=2, label='SM')
    axes[row, 1].hist(ge['probs'][ge['labels'] == 1], bins=50, density=True, histtype='step', linewidth=2, label='BSM')
    axes[row, 1].set_title(f'DeepGCN scores: {bsm_name}')
    axes[row, 1].legend(); axes[row, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('plots/gnn_ll_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()